# WhatsApp Chat Analysis
This notebook parses and analyzes the WhatsApp chat export file `WhatsApp Chat with IIITNR FRESHERS 2026.txt`.

### Key Challenges in Parsing WhatsApp Chat Logs:
1. **Unicode Space Formatting (`\u202f`):** WhatsApp logs on Apple/modern systems format the time with a narrow no-break space (`\u202f`) instead of a regular space before `AM`/`PM` (e.g. `10:26 PM`). Standard string parsing or datetime conversions will fail unless this is normalized or matched in regex.
2. **Multiline Messages:** When a message has newline characters, subsequent lines don't start with a timestamp pattern. We must track state and append continuation lines to the previous message.
3. **System Notifications vs. User Messages:** System notifications (like "Messages and calls are encrypted" or "Sundeep created group...") do not follow the standard `Sender: Message` format.

In [11]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def parse_whatsapp_chat(filepath):
    # Regex to match the timestamp: e.g., "4/19/26, 10:26 PM - "
    # Uses [\s\u202f] to match both regular spaces and narrow no-break spaces.
    pattern = r'^(\d{1,2}/\d{1,2}/\d{2,4},\s+\d{1,2}:\d{2}[\s\u202f]?[AP]M)\s+-\s+(.*)$'
    
    parsed_data = []
    current_date = None
    current_sender = None
    current_message = []
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip('\n')
            match = re.match(pattern, line)
            
            if match:
                # Append the previous message if exists
                if current_date:
                    parsed_data.append({
                        'Date': current_date,
                        'Sender': current_sender,
                        'Message': '\n'.join(current_message)
                    })
                
                date_str, rest = match.groups()
                current_date = date_str.replace('\u202f', ' ') # normalize Unicode space
                
                # Check for sender vs system message
                if ': ' in rest:
                    current_sender, message = rest.split(': ', 1)
                    # Clean up sender name
                    current_sender = current_sender.replace('~\u202f', '').replace('~', '').strip()
                    current_message = [message]
                else:
                    current_sender = 'System'
                    current_message = [rest]
            else:
                # Continuation of the previous message
                if current_date:
                    current_message.append(line)
                    
        # Append final message
        if current_date:
            parsed_data.append({
                'Date': current_date,
                'Sender': current_sender,
                'Message': '\n'.join(current_message)
            })
            
    df = pd.DataFrame(parsed_data)
    df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%y, %I:%M %p')
    return df

# Load and parse the chat log
chat_file = "WhatsApp Chat with IIITNR FRESHERS 2026.txt"
df = parse_whatsapp_chat(chat_file)
df

,Date,Sender,Message
0,2026-04-19 22:26:00,System,Messages and calls are end-to-end encrypted. O...
1,2026-04-19 11:10:00,System,"~ Sundeep Kumar Panigrahi created group ""IIITN..."
2,2026-04-19 22:26:00,System,You joined using this group's invite link
3,2026-04-19 22:37:00,System,~ Chinmay Chandak added ~ CHIRAG BHOYAR
4,2026-04-19 23:15:00,System,~ Teju.. was added
...,...,...,...
3729,2026-06-09 22:33:00,+91 81442 55926,i guess
3730,2026-06-09 22:34:00,+91 99583 41037,
3731,2026-06-09 22:45:00,Neelabjo Dsai,Sab aaye the🫡 @⁨~Satyam⁩
3732,2026-06-09 23:58:00,+91 84708 76003,Achaa


In [17]:
df["Day"] = df["Date"].dt.day
df["month"] = df["Date"].dt.month_name()
df["hour"] = df["Date"].dt.hour
df["minute"] = df["Date"].dt.minute

df

,Date,Sender,Message,Day,month,hour,minute
0,2026-04-19 22:26:00,System,Messages and calls are end-to-end encrypted. O...,19,April,22,26
1,2026-04-19 11:10:00,System,"~ Sundeep Kumar Panigrahi created group ""IIITN...",19,April,11,10
2,2026-04-19 22:26:00,System,You joined using this group's invite link,19,April,22,26
3,2026-04-19 22:37:00,System,~ Chinmay Chandak added ~ CHIRAG BHOYAR,19,April,22,37
4,2026-04-19 23:15:00,System,~ Teju.. was added,19,April,23,15
...,...,...,...,...,...,...,...
3729,2026-06-09 22:33:00,+91 81442 55926,i guess,9,June,22,33
3730,2026-06-09 22:34:00,+91 99583 41037,,9,June,22,34
3731,2026-06-09 22:45:00,Neelabjo Dsai,Sab aaye the🫡 @⁨~Satyam⁩,9,June,22,45
3732,2026-06-09 23:58:00,+91 84708 76003,Achaa,9,June,23,58
